ASL Dataset from Kaggle - https://www.kaggle.com/datasets/debashishsau/aslamerican-sign-language-aplhabet-dataset/data

Using the Hands model:
- loop through dataset
- use model to detect hands and output coordinates

After I've produced a CSV with labled classes of different individual letters in ASL:
- pass through classifier model to train recognition of individual ASL letters
- maybe it'll work??

In [ ]:
# imports
import cv2
import mediapipe as mp
import numpy as np
import csv
import os
from tqdm import tqdm # for visualizing progress while running Hands through dataset

# drawing and hand utils
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
mp_hands_connections = mp.solutions.hands_connections

# training dataset
dataset_path = "ASL_Alphabet_Dataset/asl_alphabet_train"

In [ ]:
# csv for detected hand coordinates
csv_file = "asl_landmarks.csv"

with mp_hands.Hands( # initialize hand model
  static_image_mode=True, # for static images
  max_num_hands=2,
  min_detection_confidence=0.5) as hands:
    
    # open csv
    with open(csv_file, mode='w', newline='') as f:
      writer = csv.writer(f)

      # header makes csv easier to read
      header = ['label'] + [f'x{i}' for i in range(21)] + [f'y{i}' for i in range(21)] + [f'z{i}' for i in range(21)]
      writer.writerow(header)

      # loop through each class (letter) folder
      for label in os.listdir(dataset_path):
          label_path = os.path.join(dataset_path, label)
          if not os.path.isdir(label_path):
            continue  # skip stray files in case any exist

          # loop through images in each folder, use tqdm to see progress
          for img_file in tqdm(os.listdir(label_path), desc=f"Processing {label}"):
            img_path = os.path.join(label_path, img_file)

            # read image, convert from BGR to RGB (mediapipe takes RGB)
            image = cv2.imread(img_path)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # detect hand
            results = hands.process(image_rgb)

            # skip if there isn't a hand detectable
            if not results.multi_hand_landmarks:
                continue

            # loop through all detected hands
            for hand_landmarks in results.multi_hand_landmarks:
              landmark_list = []

              # extract position (x, y, z) for all 21 hand landmarks
              for lm in hand_landmarks.landmark:
                landmark_list.extend([lm.x, lm.y, lm.z])  # normalized 0-1

              # add label at start
              row = [label] + landmark_list

              # write
              writer.writerow(row)

In [ ]:
import pandas as pd

# check how many lines per letter
df = pd.read_csv("asl_landmarks.csv")

class_counts = df['label'].value_counts()
print(class_counts)

In [ ]:
# i should delete the nothing class, since i'm not training a cnn from scratch
import pandas as pd
from sklearn.model_selection import train_test_split

# Load CSV
df = pd.read_csv("asl_landmarks.csv")

# Drop rows where label == "nothing"
df = df[df["label"] != "nothing"]

# Save cleaned CSV
df.to_csv("asl_landmarks_clean.csv", index=False)

# load csv
df = pd.read_csv("asl_landmarks_clean.csv")

# first col is letter, rest are features
X = df.iloc[:, 1:]  # features
y = df.iloc[:, 0]   # labels (letters)

# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

clf = RandomForestClassifier(n_estimators=400, random_state=42, verbose=2)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import pickle

# save model
with open("asl_letter_classifier.pkl", "wb") as f:
    pickle.dump(clf, f)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pickle

# Load trained ASL model
with open("asl_letter_classifier.pkl", "rb") as f:
    clf = pickle.load(f)

# Open webcam (0 is usually built-in MacBook cam)
cap = cv2.VideoCapture(0)

# MediaPipe setup
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.5
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip for mirror view
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb_frame)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            # Extract coordinates
            landmarks = []
            for lm in hand_landmarks.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            # Convert to numpy array, reshape for prediction
            X_input = np.array(landmarks).reshape(1, -1)

            # Predict letter
            pred = clf.predict(X_input)[0]

            # Draw landmarks
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            # Get wrist coordinates for placing text
            h, w, _ = frame.shape
            wrist = hand_landmarks.landmark[0]
            wrist_x, wrist_y = int(wrist.x * w), int(wrist.y * h)

            # Draw prediction text near the wrist
            cv2.putText(frame, f'Predicted: {pred}',
                        (wrist_x + 20, wrist_y - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 1,
                        (0, 255, 0), 2)

    cv2.imshow("ASL Classifier", frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

In [ ]:
# not that accurate yet, i'll try parsing through dataset again. modify to not parse through "nothing" label
# imports
import cv2
import mediapipe as mp
import numpy as np
import csv
import os
from tqdm import tqdm # for visualizing progress while running Hands through dataset

# drawing and hand utils
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
mp_hands_connections = mp.solutions.hands_connections

# training dataset
dataset_path = "ASL_Alphabet_Dataset/asl_alphabet_train"


# csv for detected hand coordinates
csv_file = "asl_landmarks_2.csv"

with mp_hands.Hands( # initialize hand model
  static_image_mode=True, # for static images
  max_num_hands=2,
  min_detection_confidence=0.65) as hands: # raised confidence to 0.65 - lose more images, but higher quality coordinates
    
    # open csv
    with open(csv_file, mode='w', newline='') as f:
      writer = csv.writer(f)

      # header makes csv easier to read
      header = ['label'] + [f'x{i}' for i in range(21)] + [f'y{i}' for i in range(21)] + [f'z{i}' for i in range(21)]
      writer.writerow(header)


      # loop through each class (letter) folder
      for label in os.listdir(dataset_path):
        
        if label == "nothing":
            continue # skip nothing class (since Hands shouldn't even detect hand)

        label_path = os.path.join(dataset_path, label)
        if not os.path.isdir(label_path):
          continue  # skip stray files in case any exist

        # first time doing it, 199k files only became 150k lines in csv.
        # i ran it, and it makes sense since at 0.5 confidence, i lose around 25% of images per class.
        # raised confidence to 0.65, see if it'll make a smaller but more accurate dataset
        total_images = 0
        skipped_images = 0
        written_rows = 0

        # loop through images in each folder, use tqdm to see progress
        for img_file in tqdm(os.listdir(label_path), desc=f"Processing {label}"):
          total_images += 1
          img_path = os.path.join(label_path, img_file)

          # read image, convert from BGR to RGB (mediapipe takes RGB)
          image = cv2.imread(img_path)
          image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

          # detect hand
          results = hands.process(image_rgb)

          # skip if there isn't a hand detectable
          if not results.multi_hand_landmarks:
              skipped_images += 1
              continue

          # loop through all detected hands
          for hand_landmarks in results.multi_hand_landmarks:
            landmark_list = []

            # extract position (x, y, z) for all 21 hand landmarks
            for lm in hand_landmarks.landmark:
              landmark_list.extend([lm.x, lm.y, lm.z])  # normalized 0-1

            # add label at start
            row = [label] + landmark_list

            # write
            writer.writerow(row)
            written_rows += 1

        print(f"Label {label}: Total={total_images}, Skipped={skipped_images}, Written={written_rows}") # write per label


In [ ]:
import pandas as pd

# check how many lines per letter
df = pd.read_csv("asl_landmarks_2.csv")

class_counts = df['label'].value_counts()
print(class_counts)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# load csv
df = pd.read_csv("asl_landmarks_2.csv")

# first col is letter, rest are features
X = df.iloc[:, 1:]  # features
y = df.iloc[:, 0]   # labels (letters)

# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

clf = RandomForestClassifier(n_estimators=600, random_state=42, verbose=2)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
import pickle

# save model
with open("asl_letter_classifier_2.pkl", "wb") as f:
    pickle.dump(clf, f)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pickle

# Load trained ASL model
with open("asl_letter_classifier_2.pkl", "rb") as f:
    clf = pickle.load(f)

# Open webcam (0 is usually built-in MacBook cam)
cap = cv2.VideoCapture(0)

# MediaPipe setup
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.65,
    min_tracking_confidence=0.65
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip for mirror view
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb_frame)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            # Extract coordinates
            landmarks = []
            for lm in hand_landmarks.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])

            # Convert to numpy array, reshape for prediction
            X_input = np.array(landmarks).reshape(1, -1)

            # Predict letter
            pred = clf.predict(X_input)[0]

            # Draw landmarks
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            # Get wrist coordinates for placing text
            h, w, _ = frame.shape
            wrist = hand_landmarks.landmark[0]
            wrist_x, wrist_y = int(wrist.x * w), int(wrist.y * h)

            # Draw prediction text near the wrist
            cv2.putText(frame, f'Predicted: {pred}',
                        (wrist_x + 20, wrist_y - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 1,
                        (0, 255, 0), 2)

    cv2.imshow("ASL Classifier", frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("asl_landmarks_2.csv")

X = df.iloc[:, 1:]
y = df.iloc[:, 0]

# encode labels (letters)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# d matrix
dtrain = xgb.DMatrix(X_train_sub, label=y_train_sub)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test)

params = {
    "objective": "multi:softmax",
    "num_class": len(le.classes_),
    "max_depth": 5,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "min_child_weight": 2,
    "gamma": 1,
    "eval_metric": "mlogloss",
}

evals = [(dtrain, "train"), (dval, "val")]

# train
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=400,
    evals=evals,
    early_stopping_rounds=20
)

# predict
y_pred = bst.predict(dtest)
y_pred_letters = le.inverse_transform(y_pred.astype(int))

# evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

[0]	train-mlogloss:2.62743	val-mlogloss:2.63958
[1]	train-mlogloss:2.30183	val-mlogloss:2.31939
[2]	train-mlogloss:2.06476	val-mlogloss:2.08665
[3]	train-mlogloss:1.88156	val-mlogloss:1.90687
[4]	train-mlogloss:1.72457	val-mlogloss:1.75200
[5]	train-mlogloss:1.59052	val-mlogloss:1.62035
[6]	train-mlogloss:1.47845	val-mlogloss:1.51061
[7]	train-mlogloss:1.38157	val-mlogloss:1.41546
[8]	train-mlogloss:1.29466	val-mlogloss:1.32992
[9]	train-mlogloss:1.21349	val-mlogloss:1.24985
[10]	train-mlogloss:1.14335	val-mlogloss:1.18021
[11]	train-mlogloss:1.07790	val-mlogloss:1.11536
[12]	train-mlogloss:1.01909	val-mlogloss:1.05686
[13]	train-mlogloss:0.96373	val-mlogloss:1.00200
[14]	train-mlogloss:0.91381	val-mlogloss:0.95267
[15]	train-mlogloss:0.86822	val-mlogloss:0.90742
[16]	train-mlogloss:0.82582	val-mlogloss:0.86498
[17]	train-mlogloss:0.78673	val-mlogloss:0.82617
[18]	train-mlogloss:0.75136	val-mlogloss:0.79084
[19]	train-mlogloss:0.71614	val-mlogloss:0.75567
[20]	train-mlogloss:0.68446	va

In [8]:
# save model
bst.save_model("xgb_asl_model.json")

# save label encoder
import pickle
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [14]:
import xgboost as xgb
import pickle
import numpy as np
import cv2
import mediapipe as mp

# Load model
bst = xgb.Booster()
bst.load_model("xgb_asl_model.json")

# Load label encoder
with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.65,
    min_tracking_confidence=0.65
)
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Flip for mirror view
    frame = cv2.flip(frame, 1)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Process hand landmarks
    result = hands.process(rgb_frame)
    
    if result.multi_hand_landmarks:
        hand_landmarks = result.multi_hand_landmarks[0]
        mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
        
        # Flatten landmarks to match CSV features
        landmark_list = []
        for lm in hand_landmarks.landmark:
            landmark_list.extend([lm.x, lm.y, lm.z])

        X_input = np.array(landmark_list).reshape(1, -1)

        # Use the same feature names as your CSV
        feature_names = X_train_sub.columns.tolist()
        dinput = xgb.DMatrix(X_input, feature_names=feature_names)

        pred = bst.predict(dinput)
        letter = le.inverse_transform(pred.astype(int))[0]
                
        # Get wrist coordinates for placing text
        h, w, _ = frame.shape
        wrist = hand_landmarks.landmark[0]
        wrist_x, wrist_y = int(wrist.x * w), int(wrist.y * h)

        # Draw prediction text near the wrist
        cv2.putText(frame, f'Predicted: {letter}',
                    (wrist_x + 20, wrist_y - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0, 255, 0), 2)
    
    cv2.imshow("ASL Prediction", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

I0000 00:00:1759290267.755367 19453072 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4 Pro
W0000 00:00:1759290267.760490 19551029 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1759290267.764923 19551029 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


-1

In [15]:
# testing different params to see how to prevent overfitting

from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

param_grid = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "n_estimators": [100, 200, 400],
    "subsample": [0.7, 0.8, 1],
    "colsample_bytree": [0.7, 0.8, 1],
    "gamma": [0, 1, 5],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 1.5, 2],
}

xgb_clf = XGBClassifier(
    objective="multi:softmax",
    num_class=len(le.classes_),
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)

grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,  # 3-fold cross-validation
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train) # create combo selecting best parameters
print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation accuracy: ", grid_search.best_score_)

# use on test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Fitting 3 folds for each of 6561 candidates, totalling 19683 fits


Traceback (most recent call last):
  File "/opt/anaconda3/envs/asl-env/lib/python3.10/runpy.py", line 187, in _run_module_as_main
    mod_name, mod_spec, code = _get_module_details(mod_name, _Error)
  File "/opt/anaconda3/envs/asl-env/lib/python3.10/runpy.py", line 110, in _get_module_details
    __import__(pkg_name)
  File "/opt/anaconda3/envs/asl-env/lib/python3.10/site-packages/joblib/__init__.py", line 114, in <module>
    from ._cloudpickle_wrapper import wrap_non_picklable_objects
  File "/opt/anaconda3/envs/asl-env/lib/python3.10/site-packages/joblib/_cloudpickle_wrapper.py", line 14, in <module>
    from .externals.loky import wrap_non_picklable_objects
  File "/opt/anaconda3/envs/asl-env/lib/python3.10/site-packages/joblib/externals/loky/__init__.py", line 20, in <module>
    from .backend.reduction import set_loky_pickler
  File "/opt/anaconda3/envs/asl-env/lib/python3.10/site-packages/joblib/externals/loky/backend/reduction.py", line 82, in <module>
    from joblib.externals

KeyboardInterrupt: 